# CO_baseline_comparison
**Distribution-Free Recalibration of Tail Quantile Forecasts under Temporal Dependence**

Pele, D.T., Lessmann, S., Härdle, W.K. (2026)

Compares conformal recalibration against four alternative post-hoc correction methods:
scale correction, historical quantile, quantile regression, and isotonic regression.

**Repository:** [QuantLet/Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──
DATA_DIR = Path('/Users/danielpele/Documents/2026 CFP LLM VaR/cfp_ijf_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('../../cfp_ijf_data')  # fallback for relative path

ASSETS = [
    'ASX200', 'AUDUSD', 'BOVESPA', 'BTC', 'FCHI', 'CBU0', 'DJCI', 'ETH',
    'EURUSD', 'FTSE100', 'GBPUSD', 'GDAXI', 'GOLD', 'HSI', 'IBGL', 'ICLN',
    'NATGAS', 'NIFTY', 'NIKKEI', 'SP500', 'STOXX', 'TLT', 'USDJPY', 'WTI',
]

TSFM_MODELS = {
    'Chronos-Small': 'chronos_small',
    'Chronos-Mini':  'chronos_mini',
    'TimesFM-2.5':   'timesfm25',
    'Moirai-Small':  'moirai2',
    'Lag-Llama':     'lagllama',
}

BENCHMARK_MODELS = {
    'GJR-GARCH': 'gjr_garch',
    'GARCH-N':   'garch_n',
    'HS':        'hs',
    'EWMA':      'ewma',
}

ALL_MODELS = {**TSFM_MODELS, **BENCHMARK_MODELS}

ALPHA = 0.01
F_CAL = 0.70

print(f'DATA_DIR: {DATA_DIR}')
print(f'Assets: {len(ASSETS)}, Models: {len(ALL_MODELS)}')
print(f'alpha = {ALPHA}, F_CAL = {F_CAL}')

## Data Loading

In [ ]:
def load_pair(model_name, asset, alpha=0.01):
    """Load aligned (returns, VaR) for a given model and asset.

    Returns
    -------
    r : np.ndarray  — daily log-returns
    v : np.ndarray  — VaR forecasts (negative = loss threshold)
    """
    # Load returns
    ret_path = DATA_DIR / 'returns' / f'{asset}.csv'
    ret_df = pd.read_csv(ret_path, index_col=0, parse_dates=True)
    ret_df.columns = [c.strip().lower() for c in ret_df.columns]
    r_series = ret_df.iloc[:, 0]  # first column = log_return

    # Load VaR forecasts
    subdir = ALL_MODELS[model_name]
    if model_name in TSFM_MODELS:
        var_path = DATA_DIR / subdir / f'{asset}.parquet'
    else:
        var_path = DATA_DIR / 'benchmarks' / f'{asset}_{subdir}.parquet'

    var_df = pd.read_parquet(var_path)
    var_df.index = pd.to_datetime(var_df.index)
    col = f'VaR_{alpha}'
    v_series = var_df[col]

    # Align on common dates
    common = r_series.index.intersection(v_series.index)
    common = common.sort_values()
    r = r_series.loc[common].values.astype(float)
    v = v_series.loc[common].values.astype(float)

    # Drop NaNs
    mask = ~(np.isnan(r) | np.isnan(v))
    return r[mask], v[mask]

# Quick test
r_test, v_test = load_pair('Chronos-Small', 'SP500', ALPHA)
print(f'Chronos-Small / SP500: {len(r_test)} days, VaR range [{v_test.min():.4f}, {v_test.max():.4f}]')

## Correction Methods

In [ ]:
# ── Backtesting helpers ──

def kupiec_pval(n, x, alpha):
    """Kupiec (1995) proportion-of-failures likelihood ratio test."""
    if x == 0 or x == n:
        # Boundary: use one-sided binomial p-value
        return stats.binom_test(x, n, alpha) if hasattr(stats, 'binom_test') else \
               stats.binomtest(x, n, alpha).pvalue
    pihat = x / n
    lr = 2 * (x * np.log(pihat / alpha) + (n - x) * np.log((1 - pihat) / (1 - alpha)))
    return stats.chi2.sf(lr, 1)


def basel_tl(n_viol, n_test):
    """Basel traffic light: Green/Yellow/Red based on violations per 250 days."""
    v250 = n_viol * 250 / n_test
    if v250 <= 4:
        return 'Green'
    elif v250 <= 9:
        return 'Yellow'
    else:
        return 'Red'


def quantile_score(r, v, alpha):
    """Quantile score (pinball loss) — lower is better."""
    diff = r - v
    return np.mean(np.where(diff < 0, (alpha - 1) * diff, alpha * diff))


# ── Correction methods ──

def correct_conformal(r_cal, v_cal, v_test, alpha):
    """One-sided conformal VaR correction (our method)."""
    scores = v_cal - r_cal
    # Conformal ceiling quantile (Vovk et al. 2005, Prop. 2.2)
    sorted_scores = np.sort(scores)
    n_cal = len(scores)
    k = int(np.ceil((n_cal + 1) * (1 - alpha))) - 1
    k = min(k, n_cal - 1)
    qV = float(sorted_scores[k])
    return v_test - qV, qV


def correct_scale(r_cal, v_cal, v_test, alpha):
    """Scale correction: multiply VaR by alpha / pihat_cal."""
    pihat_cal = np.mean(r_cal < v_cal)
    c_star = alpha / pihat_cal if pihat_cal > 0 else 1.0
    return v_test * c_star, c_star


def correct_hist_quantile(r_cal, v_cal, v_test, alpha):
    """Historical quantile: ignore model, use empirical alpha-quantile of calibration returns."""
    hist_q = np.quantile(r_cal, alpha)
    return np.full(len(v_test), hist_q), hist_q


def correct_qr_residual(r_cal, v_cal, v_test, alpha):
    """Quantile regression residual correction: fit r ~ beta0 + beta1 * VaR."""
    def pinball_loss(params, r, v, a):
        b0, b1 = params
        pred = b0 + b1 * v
        resid = r - pred
        return np.sum(np.where(resid < 0, (a - 1) * resid, a * resid))

    res = minimize(pinball_loss, [0.0, 1.0], args=(r_cal, v_cal, alpha),
                   method='Nelder-Mead', options={'maxiter': 5000, 'xatol': 1e-8})
    b0, b1 = res.x
    return b0 + b1 * v_test, (b0, b1)


def correct_isotonic(r_cal, v_cal, v_test, alpha):
    """Isotonic regression correction: fit monotone mapping VaR_raw -> P(violation)."""
    viol_cal = (r_cal < v_cal).astype(float)
    iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
    iso.fit(v_cal, viol_cal)
    pred_prob = iso.predict(v_test)
    scale = np.where(pred_prob > 1e-6, alpha / np.clip(pred_prob, 1e-6, 1.0), 1.0)
    return v_test * scale, None


print('Correction methods and backtesting helpers defined.')

## Run Comparison

In [ ]:
rows = []
n_skip = 0

for model in ALL_MODELS:
    for asset in ASSETS:
        try:
            r, v = load_pair(model, asset, ALPHA)
            n = len(r)
            n_cal = int(n * F_CAL)
            r_cal, v_cal = r[:n_cal], v[:n_cal]
            r_test, v_test = r[n_cal:], v[n_cal:]
            n_test = len(r_test)

            methods = {}

            # 1. Raw (no correction)
            methods['Raw'] = (v_test, None)

            # 2. Conformal (our method)
            var_conf, qV = correct_conformal(r_cal, v_cal, v_test, ALPHA)
            methods['Conformal'] = (var_conf, qV)

            # 3. Scale correction
            var_sc, c_star = correct_scale(r_cal, v_cal, v_test, ALPHA)
            methods['Scale'] = (var_sc, c_star)

            # 4. Historical quantile
            var_hq, hist_q = correct_hist_quantile(r_cal, v_cal, v_test, ALPHA)
            methods['Hist-Quantile'] = (var_hq, hist_q)

            # 5. QR-Residual
            var_qr, qr_params = correct_qr_residual(r_cal, v_cal, v_test, ALPHA)
            methods['QR-Residual'] = (var_qr, qr_params)

            # 6. Isotonic
            var_iso, _ = correct_isotonic(r_cal, v_cal, v_test, ALPHA)
            methods['Isotonic'] = (var_iso, None)

            # Evaluate each method
            for method_name, (var_corr, extra) in methods.items():
                viol = int(np.sum(r_test < var_corr))
                pihat = viol / n_test
                p_kup = kupiec_pval(n_test, viol, ALPHA)
                tl = basel_tl(viol, n_test)
                qs = quantile_score(r_test, var_corr, ALPHA)
                width = np.mean(np.abs(var_corr))

                row = {
                    'model': model, 'asset': asset, 'method': method_name,
                    'pihat': pihat, 'kupiec_pval': p_kup, 'basel_tl': tl,
                    'qs': qs, 'var_width': width, 'n_test': n_test, 'n_viol': viol,
                }
                if method_name == 'Conformal':
                    row['qV'] = extra
                if method_name == 'Scale':
                    row['c_star'] = extra
                rows.append(row)

        except Exception as e:
            n_skip += 1

df = pd.DataFrame(rows)
print(f'Computed: {len(df)} rows '
      f'({df["model"].nunique()} models x {df["asset"].nunique()} assets x '
      f'{df["method"].nunique()} methods)')
if n_skip > 0:
    print(f'Skipped: {n_skip} (model, asset) pairs due to missing data')

## Results: Method Comparison

In [ ]:
summary = df.groupby('method').agg(
    mean_pihat=('pihat', 'mean'),
    green=('basel_tl', lambda x: (x == 'Green').sum()),
    yellow=('basel_tl', lambda x: (x == 'Yellow').sum()),
    red=('basel_tl', lambda x: (x == 'Red').sum()),
    kupiec_pass=('kupiec_pval', lambda x: (x > 0.05).sum()),
    mean_QS=('qs', 'mean'),
    mean_width=('var_width', 'mean'),
).round(4)

n_combos = df.groupby('method').size().iloc[0]

# Reorder methods
method_order = ['Raw', 'Scale', 'Hist-Quantile', 'QR-Residual', 'Isotonic', 'Conformal']
summary = summary.reindex([m for m in method_order if m in summary.index])

# Add percentage columns
summary['green_pct'] = (100 * summary['green'] / n_combos).round(1)
summary['kupiec_pct'] = (100 * summary['kupiec_pass'] / n_combos).round(1)

print(f'=== Baseline Comparison: alpha = {ALPHA}, {df["model"].nunique()} models x {df["asset"].nunique()} assets ===')
print(f'    ({n_combos} combinations per method)')
print()
print(summary[['mean_pihat', 'green', 'green_pct', 'red', 'kupiec_pass', 'kupiec_pct',
               'mean_QS', 'mean_width']].to_string())

## Results by Model

In [ ]:
by_model = df.groupby(['model', 'method']).agg(
    green=('basel_tl', lambda x: (x == 'Green').sum()),
    mean_QS=('qs', 'mean'),
).round(4)

pivot_green = by_model['green'].unstack('method')
pivot_green = pivot_green[[m for m in method_order if m in pivot_green.columns]]
print('=== Basel Green counts by model (out of 24 assets) ===')
print(pivot_green.to_string())
print()

pivot_qs = by_model['mean_QS'].unstack('method')
pivot_qs = pivot_qs[[m for m in method_order if m in pivot_qs.columns]]
print('=== Mean Quantile Score by model (lower is better) ===')
print(pivot_qs.to_string())

## Save Results

In [ ]:
out_dir = Path('results')
out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / 'baseline_comparison.csv', index=False)
summary.to_csv(out_dir / 'baseline_summary.csv')

print('Saved to results/')
print()
print('=' * 60)
print('KEY FINDING')
print('=' * 60)
conf_green = summary.loc['Conformal', 'green_pct'] if 'Conformal' in summary.index else 'N/A'
iso_green = summary.loc['Isotonic', 'green_pct'] if 'Isotonic' in summary.index else 'N/A'
print(f'Conformal Green: {conf_green}%')
print(f'Isotonic Green:  {iso_green}%')
print('Isotonic fragments the sparse 1% tail -> worse at extreme quantiles.')
print('The one-parameter conformal shift wins the bias-variance trade-off.')